# KM Putri Sakinah — metocean analysis (Labuan Bajo / Selat Padar)

Incident: **26 Dec 2025 ~20:30 WITA (12:30 UTC)** in Selat Padar.

This notebook orchestrates download → densify (0.01°) → spatial maps + point time series (±3 days).

**Note:** densification is for display only; native GFS is 0.25° (~28 km).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from lib.config import load_event, ensure_dirs, project_root

cfg = load_event(ROOT / "event.yaml")
ensure_dirs(cfg)
cfg["event"]["name"], cfg["time_window"], cfg["refine"]["target_deg"]

## 1. Download primary datasets (GFS + HYCOM)

Optional ERA5 / CMEMS skip if credentials are missing.

In [ ]:
from scripts import download_gfs, download_hycom, download_era5, download_cmems

download_gfs.run(ROOT / "event.yaml")
download_hycom.run(ROOT / "event.yaml")
download_era5.run(ROOT / "event.yaml")
download_cmems.run(ROOT / "event.yaml")

## 2. Spatial refine + point extract

In [ ]:
from scripts import refine_field, extract_point, plot_spatial, plot_point

refine_field.run(ROOT / "event.yaml")
extract_point.run(ROOT / "event.yaml")

## 3. Plots

Use `every=4` for a quick draft; omit / set `1` for full native cadence.

In [ ]:
plot_spatial.run(ROOT / "event.yaml", every=4)
plot_point.run(ROOT / "event.yaml")

## 4. Inspect point series near the event

In [ ]:
import pandas as pd
from IPython.display import Image, display

csv_path = ROOT / "figures/point/primary_incident_timeseries.csv"
df = pd.read_csv(csv_path, parse_dates=["time"])
event = pd.Timestamp(cfg["event"]["time_utc"])
near = df[(df["time"] >= event - pd.Timedelta("6h")) & (df["time"] <= event + pd.Timedelta("6h"))]
near.pivot_table(index="time", columns="variable", values="value")

In [ ]:
ts_png = ROOT / "figures/point/primary_incident_timeseries.png"
if ts_png.exists():
    display(Image(filename=str(ts_png)))

# Show a key spatial frame if present
key = ROOT / "figures/spatial/gfs/swh/t_2025122612.png"
if key.exists():
    display(Image(filename=str(key)))

## Limitations

- GFS 0.25° cannot resolve narrow Padar channels or island sheltering.
- 0.01° densification smooths contours for readability only.
- Reported 2–3 m waves may reflect local squalls / swell focusing not captured by the model.
- Wreck position is ~7.5 nm from the estimated sinking site (currents).